In [15]:
import os
import torch
import torch.nn as nn
import torchvision
from torchinfo import summary
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transforms = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
])

print(f"Using device {device}")

train_data_raw = torchvision.datasets.MNIST(root="mnist", train=True, download=True, transform=transforms)
test_data_raw = torchvision.datasets.MNIST(root="mnist", train=False, download=True, transform=transforms)


Using device cuda


In [17]:
os.makedirs("data", exist_ok=True)

indices = (train_data_raw.targets == 1) | (train_data_raw.targets == 2)
train_data_raw.data = train_data_raw.data[indices]
train_data_raw.targets = train_data_raw.targets[indices]

indices = (test_data_raw.targets == 1) | (test_data_raw.targets == 2)
test_data_raw.data = test_data_raw.data[indices]
test_data_raw.targets = test_data_raw.targets[indices]

if not (os.path.exists("data/train_mnist.pt") and os.path.exists("data/test_mnist.pt")):
    train_rotated_images = []
    train_labels = []
    test_rotated_images = []
    test_labels = []

    angles = [30 * i for i in range(12)]

    for img, label in tqdm(train_data_raw, desc="Creating Training Dataset"):
        for angle in angles:
            rotated = torchvision.transforms.functional.rotate(img, angle=angle)
            train_rotated_images.append(rotated)
            train_labels.append(label)

    for img, label in tqdm(test_data_raw, desc="Creating Testing Dataset"):
        for angle in angles:
            rotated = torchvision.transforms.functional.rotate(img, angle=angle)
            test_rotated_images.append(rotated)
            test_labels.append(label)

    train_rotated_images = torch.stack(train_rotated_images)
    train_labels = torch.tensor(train_labels)
    test_rotated_images = torch.stack(test_rotated_images)
    test_labels = torch.tensor(test_labels)

    torch.save({'images': train_rotated_images, 'labels': train_labels}, 'data/train_mnist.pt')
    torch.save({'images': test_rotated_images, 'labels': test_labels}, 'data/test_mnist.pt')

    print(train_rotated_images.shape)
    print(train_labels.shape)
    print(test_rotated_images.shape)
    print(test_labels.shape)
else:
    print("Dataset already exists")

Dataset already exists


In [18]:
train_data_pt = torch.load('data/train_mnist.pt')
images = train_data_pt['images']  
labels = train_data_pt['labels']

train_data = TensorDataset(images, labels)

test_data_pt = torch.load('data/test_mnist.pt')
images = test_data_pt['images']  
labels = test_data_pt['labels']

test_data = TensorDataset(images, labels)

In [19]:
batch_size = 64
train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=4)
test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=4)

In [20]:
class Encoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1, stride=1),    # (1, 28, 28) -> (32, 28, 28)
            nn.MaxPool2d(kernel_size=2, stride=2),    # (32, 28, 28) -> (32, 14, 14)
            nn.ReLU(),

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1, stride=1),    # (32, 14, 14) -> (64, 14, 14)
            nn.MaxPool2d(kernel_size=2, stride=2),    # (64, 14, 14) -> (64, 7, 7)
            nn.ReLU(),
        )

        self.mu_layer = nn.Linear(64*7*7, latent_dim)    # (1, 64*7*7) -> (1, latent_dim)
        self.logvar_layer = nn.Linear(64*7*7, latent_dim)    # (1, 64*7*7) -> (1, latent_dim)
    
    def forward(self, img):
        x = self.conv_layers(img)
        x = x.view(x.size(0), -1)    # (64, 7, 7) -> (1, 64*7*7)

        mu = self.mu_layer(x)
        logvar = self.logvar_layer(x)

        return mu, logvar

In [21]:
class Decoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()

        self.fcnn_layer = nn.Linear(latent_dim, 64*7*7)    # (1, latent_dim) -> (1, 64*7*7)

        self.conv_layers = nn.Sequential(
            nn.ConvTranspose2d(in_channels=64, out_channels=32, kernel_size=4, padding=1, stride=2),    # (64, 7, 7) -> (32, 14, 14)
            nn.ReLU(),

            nn.ConvTranspose2d(in_channels=32, out_channels=1, kernel_size=4, padding=1, stride=2),    # (32, 14, 14) -> (3, 28, 28)
            nn.Sigmoid(),
        )
    
    def forward(self, latent):
        x = self.fcnn_layer(latent)
        x = x.view(x.size(0), 64, 7, 7)    # (1, 64*7*7) -> (64, 7, 7)

        img = self.conv_layers(x)
        return img
    

In [22]:
class VAE(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)
    
    def forward(self, img):
        mu, logvar = self.encoder(img)

        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std

        out = self.decoder(z)
        return out, mu, logvar

In [23]:
class VAELoss(nn.Module):
    def __init__(self, beta):
        super().__init__()
        self.beta = beta
        self.mse_loss = nn.MSELoss(reduction="sum")
    
    def forward(self, img, out, mu, logvar):
        reconstruction_loss = self.mse_loss(img, out)
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

        loss = (reconstruction_loss + self.beta * kl_loss) / img.shape[0]
        reconstruction_loss /= img.shape[0]
        kl_loss /= img.shape[0]
        
        return loss, reconstruction_loss, kl_loss

In [24]:
def train_one_epoch(model, train_dataloader, test_dataloader, loss_func, optimizer, device):
    train_loss = 0
    train_reconstruction_loss = 0
    train_kl_loss = 0

    model.train()
    for img, label in tqdm(train_dataloader, desc="Training"):
        img = img.to(device)
        label = label.to(device)

        out, mu, logvar = model(img)

        optimizer.zero_grad()
        loss, reconstruction_loss, kl_loss = loss_func(img, out, mu, logvar)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_reconstruction_loss += reconstruction_loss.item()
        train_kl_loss += kl_loss.item()
    
    test_loss = 0
    test_reconstruction_loss = 0
    test_kl_loss = 0

    model.eval()
    with torch.no_grad():
        for img, label in tqdm(test_dataloader, desc="Validating"):
            img = img.to(device)
            label = label.to(device)

            out, mu, logvar = model(img)
            loss, reconstruction_loss, kl_loss = loss_func(img, out, mu, logvar)

            test_loss += loss.item()
            test_reconstruction_loss += reconstruction_loss.item()
            test_kl_loss += kl_loss.item()
    
    train_loss /= len(train_dataloader)  
    train_reconstruction_loss /= len(train_dataloader)  
    train_kl_loss /= len(train_dataloader)  
    test_loss /= len(test_dataloader)  
    test_reconstruction_loss /= len(test_dataloader)   
    test_kl_loss /= len(test_dataloader)

    return train_loss, train_reconstruction_loss, train_kl_loss, test_loss, test_reconstruction_loss, test_kl_loss


In [25]:
def visualize(model, test_dataloader, epoch, device):
    sample_in = []
    sample_out = []
    all_mu = []
    all_labels = []

    model.eval()
    batch = 0
    with torch.no_grad():
        for img, label in tqdm(test_dataloader, desc="Visualizing"):
            if (batch % 4 != 0):    # We only use 25% of test data for visualization
                batch += 1
                continue

            img = img.to(device)
            label = label.to(device)

            out, mu, logvar = model(img)

            if (len(sample_in) == 0):    # The first batch will be visualized for reconstruction
                sample_in.append(img)
                sample_out.append(out)
            
            all_mu.append(mu)
            all_labels.append(label)

            batch += 1
    
    sample_in = torch.cat(sample_in, dim=0)
    sample_out = torch.cat(sample_out, dim=0)
    all_mu = torch.cat(all_mu, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    fig, axes = plt.subplots(5, 2, figsize=(6, 12))
    for i in range(5):
        axes[i, 0].imshow(sample_in[i].squeeze(0).cpu())
        axes[i, 0].set_title("input")
        axes[i, 1].imshow(sample_out[i].squeeze(0).cpu())
        axes[i, 1].set_title("output")
    
    plt.tight_layout()
    plt.savefig(f"task1_results/reconstruction_epoch_{epoch}.png")
    plt.close()

    pca = PCA(n_components=2)
    all_mu_2d = pca.fit_transform(all_mu.cpu().numpy())
    
    plt.figure(figsize=(8,6)) 
    # scatter = plt.scatter(all_mu_2d[:,0], all_mu_2d[:,1], c=all_labels.cpu(), cmap="tab10",s=5)
    scatter = plt.scatter(all_mu_2d[:,0], all_mu_2d[:,1], c=all_labels.cpu(), cmap="coolwarm",s=5)

    # cbar = plt.colorbar(scatter, ticks=np.arange(10))
    # cbar.ax.set_yticklabels([f"{i}" for i in range(10)])
    cbar = plt.colorbar(scatter, ticks=[1,2])
    cbar.ax.set_yticklabels([1,2])
    plt.savefig(f"task1_results/latent_space_{epoch}.png")
    plt.close()
    


In [26]:
def train(num_epochs, model, train_dataloader, test_dataloader, loss_func, optimizer, device, patience):
    train_losses = []
    train_reconstruction_losses = []
    train_kl_losses = []
    test_losses = []
    test_reconstruction_losses = []
    test_kl_losses = []

    best_test_loss = float("inf")
    patience_count = 0
    for epoch in range(num_epochs):
        train_loss, train_reconstruction_loss, train_kl_loss, test_loss, test_reconstruction_loss, test_kl_loss = train_one_epoch(model, train_dataloader, test_dataloader, loss_func, optimizer, device)

        train_losses.append(train_loss)
        train_reconstruction_losses.append(train_reconstruction_loss)
        train_kl_losses.append(train_kl_loss)
        test_losses.append(test_loss)
        test_reconstruction_losses.append(test_reconstruction_loss)
        test_kl_losses.append(test_kl_loss)

        visualize(model, test_dataloader, epoch, device)

        print(f"Epoch {epoch} | train_loss: {train_loss:.2f} | train_recon: {train_reconstruction_loss:.2f} | train_kl: {train_kl_loss:.2f} | test_loss: {test_loss:.2f}")

        if (test_loss < best_test_loss):
            patience_count = 0
            best_test_loss = test_loss
            torch.save(model.state_dict(), "task1_results/best_model.pt")
        else:
            patience_count += 1
            if (patience_count >= patience):
                print("Early stopping")
                break

    return train_losses, train_reconstruction_losses, train_kl_losses, test_losses, test_reconstruction_losses, test_kl_losses

In [27]:
latent_dim = 64
model = VAE(latent_dim)
model = model.to(device)

summary(model, (16, 1, 28, 28))

Layer (type:depth-idx)                   Output Shape              Param #
VAE                                      [16, 1, 28, 28]           --
├─Encoder: 1-1                           [16, 64]                  --
│    └─Sequential: 2-1                   [16, 64, 7, 7]            --
│    │    └─Conv2d: 3-1                  [16, 32, 28, 28]          320
│    │    └─MaxPool2d: 3-2               [16, 32, 14, 14]          --
│    │    └─ReLU: 3-3                    [16, 32, 14, 14]          --
│    │    └─Conv2d: 3-4                  [16, 64, 14, 14]          18,496
│    │    └─MaxPool2d: 3-5               [16, 64, 7, 7]            --
│    │    └─ReLU: 3-6                    [16, 64, 7, 7]            --
│    └─Linear: 2-2                       [16, 64]                  200,768
│    └─Linear: 2-3                       [16, 64]                  200,768
├─Decoder: 1-2                           [16, 1, 28, 28]           --
│    └─Linear: 2-4                       [16, 3136]                203

In [28]:
os.makedirs("task1_results", exist_ok=True)
num_epochs = 50
beta = 1
lr = 1e-4
patience = 5
loss_func = VAELoss(beta)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

train_losses, train_reconstruction_losses, train_kl_losses, test_losses, test_reconstruction_losses, test_kl_losses = train(num_epochs, model, train_dataloader, test_dataloader, loss_func, optimizer, device, patience)

Visualizing: 100%|██████████| 407/407 [00:00<00:00, 470.90it/s]


Epoch 0 | train_loss: 49.19 | train_recon: 39.02 | train_kl: 10.16 | test_loss: 38.86


Visualizing: 100%|██████████| 407/407 [00:00<00:00, 431.55it/s]


Epoch 1 | train_loss: 35.37 | train_recon: 24.30 | train_kl: 11.07 | test_loss: 33.00


Visualizing: 100%|██████████| 407/407 [00:00<00:00, 482.54it/s]


...

Epoch 47 | train_loss: 27.29 | train_recon: 15.68 | train_kl: 11.61 | test_loss: 27.07


Visualizing: 100%|██████████| 407/407 [00:01<00:00, 397.59it/s]


Epoch 48 | train_loss: 27.28 | train_recon: 15.66 | train_kl: 11.62 | test_loss: 26.97


Visualizing: 100%|██████████| 407/407 [00:01<00:00, 400.09it/s]


Epoch 49 | train_loss: 27.25 | train_recon: 15.64 | train_kl: 11.61 | test_loss: 26.94


In [29]:
plt.figure()
plt.plot(train_losses, label="train")
plt.plot(test_losses, label="test")
plt.title("Total Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.savefig("task1_results/total_loss.png")
plt.close()

plt.figure()
plt.plot(train_kl_losses, label="train")
plt.plot(test_kl_losses, label="test")
plt.title("KL Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.savefig("task1_results/kl_loss.png")
plt.close()

plt.figure()
plt.plot(train_reconstruction_losses, label="train")
plt.plot(test_reconstruction_losses, label="test")
plt.title("Reconstruction Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.savefig("task1_results/reconstruction_loss.png")
plt.close()